# get-children-callable-param — ex3: Module.__call__ delegates to forward — and forward must raise NotImplementedError

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `get-children-callable-param`. Running the final beacon cell reports progress against the `Backprop: get_children callable param` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: get_children callable param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`get-children-callable-param`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "get-children-callable-param"
DD_SUBTOPIC = "Backprop: get_children callable param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `__call__` — the callable half of get_children / parameters / Module

Ex1 walked `__dict__` to yield child tensors. Ex2 added recursion + dotted names. The atom name is `get-children-callable-param` — the third load-bearing facet is the CALLABLE half: `nn.Module.__call__` routes through `forward`. That's why `model(x)` works without you ever writing `__call__` in your subclass.

```python
class Module:
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def forward(self, *args, **kwargs):
        raise NotImplementedError(
            f'{type(self).__name__} must implement forward'
        )
```

**Why route through `forward`, not just rename.** The full PyTorch `nn.Module.__call__` runs forward AND fires pre/post hooks, training vs eval branching, and grad-mode bookkeeping AROUND the forward call. Even our minimal version leaves the hook slot open — subclasses can override `__call__` to add behavior without touching `forward`'s subclass-overridden body.

**Why `forward` must raise NotImplementedError.** A subclass that forgets to define `forward` would silently return None on `model(x)`. Raising at the base level forces the subclass to be explicit about its compute. Same rationale as `abc.abstractmethod` — fail at the right abstraction level.

### Exercise 3 — Module.__call__ delegates to forward — and forward must raise NotImplementedError

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the nn.Module callable convention: define `__call__` on the base class to delegate to `forward`, and define `forward` itself to raise `NotImplementedError` so subclasses are forced to be explicit about their compute.
> Keywords: module, call, forward, callable, notimplemented
> ```

**KCs targeted:** `module-call-delegates-forward`, `forward-abstract-raise`

Implement the `Module` base class with TWO methods:

1. `__call__(self, *args, **kwargs)`:
   - Delegates to `self.forward(*args, **kwargs)` and returns the result.
   - Pass through all positional + keyword args verbatim.

2. `forward(self, *args, **kwargs)`:
   - Raises `NotImplementedError` with a message that includes `type(self).__name__` so the error reads something like `'MyLayer must implement forward'`.

Return the `Module` class itself from `ex3_module_class()` so the tests can subclass it.

Constraints:
- A subclass that defines `forward` MUST be callable via `instance(...)` and return whatever its `forward` returns.
- A subclass that does NOT define `forward` MUST raise `NotImplementedError` when called.
- The error message MUST contain the subclass's class name (not the literal string 'Module').

In [ ]:
def ex3_module_class():
    class Module:
        def __call__(self, *args, **kwargs):
            return self.forward(*args, **kwargs)

        def forward(self, *args, **kwargs):
            raise NotImplementedError(
                f'{type(self).__name__} must implement forward'
            )
    return Module


<details><summary>Solution</summary>

```python
def ex3_module_class():
    class Module:
        def __call__(self, *args, **kwargs):
            return self.forward(*args, **kwargs)

        def forward(self, *args, **kwargs):
            raise NotImplementedError(
                f'{type(self).__name__} must implement forward'
            )
    return Module
```

**Why split `__call__` from `forward`.** Two reasons. (a) The user writes `forward` in subclasses; calling code uses `model(x)`, which routes through `__call__`. That gives a HOOK POINT: the base class (or a subclass) can wrap `__call__` to add logging, pre/post hooks, training-vs-eval branching — without touching subclass-overridden `forward`. (b) It matches the conventional PyTorch API surface.

**`type(self).__name__`, NOT `self.__class__.__name__`.** Both work, but `type(self)` is the canonical way to get the runtime type — works correctly even when `__class__` has been monkey-patched. Same result for normal subclasses; safer in metaclass territory.

**Why raise instead of returning silently.** A subclass that forgot `forward` returning None silently would propagate garbage through the model. Raising at the base level forces the subclass author to be explicit — fail loud at the right abstraction level. Same pattern as `abc.abstractmethod`.

**This is the third facet of the atom.** Ex1 walked `__dict__` for children (the 'get-children' half). Ex2 recursed for `parameters()`. Ex3 covers the 'callable' half — the atom name is literally `get-children-callable-param`, and `__call__` delegation is what makes a Module a callable parameterized function instead of just a container.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()